In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/safalsingh/quant-singularity-data/finetune_instructions.jsonl
/kaggle/input/datasets/safalsingh/quant-singularity-data/market_states.parquet
/kaggle/input/datasets/safalsingh/quant-singularity-data/retrieve.py
/kaggle/input/datasets/safalsingh/quant-singularity-data/finetune_instructions_clean.jsonl
/kaggle/input/datasets/safalsingh/quant-singularity-data/rag_corpus.jsonl


**Train**

In [2]:
import os
os.environ["PYTHONUTF8"] = "1"

In [3]:
# Install deps
import subprocess, sys
def pip(*p): subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *p])
pip("transformers>=4.40", "peft>=0.9", "trl>=0.8", "bitsandbytes>=0.43",
    "accelerate>=0.27", "datasets>=2.18", "mlflow>=2.11")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 105.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 106.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8

In [4]:
import os
os.environ["PYTHONUTF8"] = "1"


import json, warnings, re, math, uuid
import numpy as np
import pandas as pd
from datetime import datetime, timezone
from pathlib import Path

import torch
import mlflow

from datasets import Dataset

from peft import LoraConfig, TaskType, get_peft_model, PeftModel

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)

from trl import SFTTrainer

warnings.filterwarnings("ignore")
DATA_DIR    = Path("/kaggle/input/datasets/safalsingh/quant-singularity-data")
OUTPUT_DIR  = Path("/kaggle/working")

ADAPTER_DIR = OUTPUT_DIR / "adapter"
ADAPTER_DIR.mkdir(exist_ok=True)

BASE_MODEL  = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

VALID_DIRS  = {"CE", "PE", "NEUTRAL"}
VALID_HORIZ = {"intraday", "next_session"}

ADX_THRESH  = 20.0
CONV_THRESH = 0.40

mlflow.set_tracking_uri((OUTPUT_DIR / "mlruns").as_uri())
mlflow.set_experiment("quant-singularity-signal-pod")


2026/05/08 08:54:34 INFO mlflow.tracking.fluent: Experiment with name 'quant-singularity-signal-pod' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///kaggle/working/mlruns/764293315480343089', creation_time=1778230474620, experiment_id='764293315480343089', last_update_time=1778230474620, lifecycle_stage='active', name='quant-singularity-signal-pod', tags={}, trace_location=None, workspace='default'>

In [5]:
# Load clean data
rows = [json.loads(l) for l in
        (DATA_DIR / "finetune_instructions_clean.jsonl").read_text("utf-8").splitlines() if l.strip()]
print(f"Rows: {len(rows)}")

def fmt(r):
    return f"### Instruction:\n{r['instruction']}\n\n### Input:\n{r['input']}\n\n### Response:\n{r['output']}"

texts = [fmt(r) for r in rows]
n_val = max(1, int(0.10 * len(texts)))
train_ds = Dataset.from_dict({"text": texts[:-n_val]})
val_ds   = Dataset.from_dict({"text": texts[-n_val:]})
print(f"Train {len(train_ds)}  Val {len(val_ds)}")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, dtype=torch.float16, device_map="auto"
)
model.config.use_cache = False

lora_cfg = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, bias="none",
                      task_type=TaskType.CAUSAL_LM,
                      target_modules=["q_proj","v_proj"])
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=512, padding=False)

train_tok = train_ds.map(tokenize, batched=True, remove_columns=["text"])
val_tok   = val_ds.map(tokenize,   batched=True, remove_columns=["text"])
collator  = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

train_args = TrainingArguments(
    output_dir=str(ADAPTER_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    weight_decay=0.01,
    fp16=True,
    bf16=False,
    logging_steps=5,
    save_strategy="epoch",
    eval_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    report_to="none",
    max_grad_norm=1.0,
    dataloader_num_workers=2,
    optim="adamw_torch",
)

with mlflow.start_run(run_name="kaggle_t4_r8") as run:
    mlflow.log_params({"base_model": BASE_MODEL, "lora_rank": 8,
                       "train_samples": len(train_ds), "epochs": 3})
    trainer = SFTTrainer(model=model, args=train_args,
                         train_dataset=train_tok, eval_dataset=val_tok,
                         data_collator=collator)
    result = trainer.train()
    mlflow.log_metric("train_loss", result.training_loss)
    trainer.save_model(str(ADAPTER_DIR))
    tokenizer.save_pretrained(str(ADAPTER_DIR))
    print(f"Done. Run ID: {run.info.run_id}")
    print(f"Adapter at: {ADAPTER_DIR}")


Rows: 261
Train 235  Val 26


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

Map:   0%|          | 0/235 [00:00<?, ? examples/s]

Map:   0%|          | 0/26 [00:00<?, ? examples/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Epoch,Training Loss,Validation Loss
1,1.740515,1.612937
2,1.167369,1.098291
3,0.971997,0.974155


Done. Run ID: f0a11051b56d44e790bb9d1ad3aef18c
Adapter at: /kaggle/working/adapter


save adapter

**eval cell**

cecell a blocks# cell blocks



In [6]:
"""
KAGGLE CELL A — Comprehensive Eval
Paste this as a new cell in your Kaggle notebook after training.
Covers: 5-day rolling windows, Wilson CI, VIX regime slicing,
        conviction ECE calibration, orchestrator decision log.
"""

import json, re, math, uuid, numpy as np, pandas as pd
from datetime import datetime, timezone
from pathlib import Path
from collections import defaultdict, Counter
import torch

# ── Paths ─────────────────────────────────────────────────────────────────
ADAPTER_DIR = Path("/kaggle/working/adapter")
DATA_DIR    = Path("/kaggle/input/datasets/safalsingh/quant-singularity-data")
BASE_MODEL  = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
VALID_DIRS  = {"CE","PE","NEUTRAL"}
VALID_HORIZ = {"intraday","next_session"}
ADX_THRESH  = 20.0
CONV_THRESH = 0.40
LOG_FILE    = Path("/kaggle/working/orchestrator_decisions.jsonl")


In [7]:
# ── Load model (reuse if already in memory from training cell) ────────────
try:
    _ = model
    print("Reusing model from training cell.")
except NameError:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel
    print("Loading model + adapter...")
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=torch.float16, device_map="auto")
    model = PeftModel.from_pretrained(base, str(ADAPTER_DIR))
    model.eval()
    print("Done.")

Reusing model from training cell.


In [8]:
# ── Wilson score CI ───────────────────────────────────────────────────────
def wilson_ci(k, n, z=1.96):
    if n == 0: return (float("nan"), float("nan"))
    p = k/n
    denom = 1 + z**2/n
    c = (p + z**2/(2*n)) / denom
    m = z * math.sqrt(p*(1-p)/n + z**2/(4*n**2)) / denom
    return (round(max(0,c-m),4), round(min(1,c+m),4))

In [9]:
# ── Conviction ECE ────────────────────────────────────────────────────────
def ece(convictions, correct_flags, n_bins=5):
    bins = np.linspace(0,1,n_bins+1)
    score, n = 0.0, len(convictions)
    for lo,hi in zip(bins[:-1],bins[1:]):
        mask = [lo<=c<hi for c in convictions]
        if not any(mask): continue
        bc = [c for c,m in zip(convictions,mask) if m]
        ba = [a for a,m in zip(correct_flags,mask) if m]
        score += (len(bc)/n)*abs(np.mean(bc)-np.mean(ba))
    return round(score,4)

In [10]:
# ── Pod inference ─────────────────────────────────────────────────────────
def predict_pod(ms):
    inp = json.dumps({k: ms.get(k) for k in
        ["nifty_spot","atm_iv","iv_skew_25d","pcr","adx_14",
         "realized_vol_5d","vix_india","dte_nearest","moneyness_band"]})
    prompt = (
        "### Instruction:\nYou are a NIFTY 50 options signal generator. "
        "Return ONLY valid JSON with keys: direction (CE/PE/NEUTRAL), "
        "conviction (float 0-1), horizon (intraday/next_session), "
        "signal_id (string), generated_at (string).\n\n"
        f"### Input:\n{inp}\n\n### Response:\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=128, temperature=0.1,
                             do_sample=True, pad_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:],
                            skip_special_tokens=True).strip()
    # Try full JSON parse
    clean = re.sub(r"```(?:json)?\s*","",text).strip()
    for c in [clean, *re.findall(r'\{[^{}]*\}', clean, re.DOTALL)]:
        try:
            o = json.loads(c.strip())
            if o.get("direction") in VALID_DIRS and o.get("horizon") in VALID_HORIZ:
                conv = float(o.get("conviction",0))
                if 0<=conv<=1:
                    o["conviction"] = conv
                    return o, "parsed"
        except: pass
    # Regex fallback for truncated JSON
    dm = re.search(r'"direction"\s*:\s*"(CE|PE|NEUTRAL)"', text)
    cm = re.search(r'"conviction"\s*:\s*(\d*\.?\d+)', text)
    hm = re.search(r'"horizon"\s*:\s*"(intraday|next_session)"', text)
    if dm and cm and hm:
        return {"direction":dm.group(1),"conviction":float(cm.group(1)),
                "horizon":hm.group(1),"signal_id":str(uuid.uuid4()),
                "generated_at":datetime.now(timezone.utc).isoformat()}, "recovered"
    return {"direction":"NEUTRAL","conviction":0.0,"horizon":"intraday",
            "signal_id":str(uuid.uuid4()),
            "generated_at":datetime.now(timezone.utc).isoformat(),
            "_fallback":True}, "fallback"

In [11]:
# ── Orchestrator ──────────────────────────────────────────────────────────
def orchestrate(ms, pod_sig, parse_status):
    adx  = float(ms.get("adx_14",0))
    conv = float(pod_sig.get("conviction",0))
    now  = datetime.now(timezone.utc).isoformat()

    if adx < ADX_THRESH:
        action = "SUPPRESSED_ADX"
        values = {"adx_14": round(adx,2), "threshold": ADX_THRESH}
        final  = {"direction":"NEUTRAL","conviction":0.0,"horizon":"intraday",
                  "signal_id":str(uuid.uuid4()),"generated_at":now}
    elif parse_status == "fallback":
        action = "SUPPRESSED_PARSE"
        values = {"reason": "model_fallback"}
        final  = {"direction":"NEUTRAL","conviction":0.0,"horizon":"intraday",
                  "signal_id":str(uuid.uuid4()),"generated_at":now}
    elif conv < CONV_THRESH:
        action = "DOWNGRADED_CONVICTION"
        values = {"conviction": round(conv,4), "threshold": CONV_THRESH}
        final  = {**pod_sig, "direction":"NEUTRAL"}
    else:
        action = "PASS_THROUGH"
        values = {"conviction": round(conv,4), "adx_14": round(adx,2)}
        final  = dict(pod_sig)

    # Log every decision with reason code + triggering values
    log_rec = {"ts":now,"action":action,"values":values,
               "pod_direction":pod_sig.get("direction"),
               "pod_conviction":round(conv,4),
               "final_direction":final["direction"],
               "parse_status":parse_status,
               "market_state":{k:ms.get(k) for k in
                   ["adx_14","vix_india","pcr","dte_nearest","moneyness_band"]}}
    with open(LOG_FILE,"a") as f: f.write(json.dumps(log_rec)+"\n")

    final["orchestrator_action"] = action
    final["orchestrator_values"] = values
    return final


In [12]:
# ── Data audit summary (display before eval) ─────────────────────────────
print("=" * 58)
print("DATA AUDIT SUMMARY")
print("=" * 58)
raw_lines = (DATA_DIR/"finetune_instructions.jsonl").read_text("utf-8").splitlines()
clean_lines= (DATA_DIR/"finetune_instructions_clean.jsonl").read_text("utf-8").splitlines()
print(f"Raw instruction rows:        {len(raw_lines)}")
print(f"Post-audit clean rows:       {len(clean_lines)}")
print(f"Excluded (uncorrectable):    {len(raw_lines)-len(clean_lines)}")
conviction_types = {"numeric":0,"string":0}
for l in raw_lines:
    try:
        o = json.loads(json.loads(l)["output"])
        float(o["conviction"])
        conviction_types["numeric"] += 1
    except: conviction_types["string"] += 1
print(f"Conviction numeric/string:   {conviction_types['numeric']}/{conviction_types['string']}")
clean_rows = [json.loads(l) for l in clean_lines if l.strip()]
dirs_train = Counter(json.loads(r["output"])["direction"] for r in clean_rows)
print(f"Training label distribution: {dict(dirs_train)}")

# ── Load eval data (days 31-60) ───────────────────────────────────────────
df = pd.read_parquet(DATA_DIR/"market_states.parquet").sort_values("timestamp").reset_index(drop=True)
df["date"] = pd.to_datetime(df["timestamp"]).dt.date
unique_days = sorted(df["date"].unique())
eval_days   = set(unique_days[30:60])
eval_df     = df[df["date"].isin(eval_days)].copy()
print(f"\nEval rows: {len(eval_df)}  (days 31-60)")
print(f"Eval label distribution: {dict(Counter(eval_df['label']))}")


DATA AUDIT SUMMARY
Raw instruction rows:        300
Post-audit clean rows:       261
Excluded (uncorrectable):    39
Conviction numeric/string:   255/45
Training label distribution: {'PE': 70, 'NEUTRAL': 116, 'CE': 75}

Eval rows: 390  (days 31-60)
Eval label distribution: {'NEUTRAL': 151, 'PE': 140, 'CE': 99}


In [13]:
# ── Run eval ──────────────────────────────────────────────────────────────
LOG_FILE.write_text("")   # clear log for this run
labels, final_dirs, actions, convictions, vix_vals, dates_list = [],[],[],[],[],[]
parse_statuses = []

for i,(_, row) in enumerate(eval_df.iterrows()):
    ms = row.to_dict()
    ms.pop("date", None)
    label = ms.pop("label", None)
    pod_sig, pstatus = predict_pod(ms)
    final = orchestrate(ms, pod_sig, pstatus)
    labels.append(label)
    final_dirs.append(final["direction"])
    actions.append(final["orchestrator_action"])
    convictions.append(float(final.get("conviction",0)))
    vix_vals.append(float(row["vix_india"]))
    dates_list.append(row["date"])
    parse_statuses.append(pstatus)
    if i%50==0: print(f"  {i}/{len(eval_df)} done...")

print(f"  {len(eval_df)}/{len(eval_df)} done.")


  0/390 done...
  50/390 done...
  100/390 done...
  150/390 done...
  200/390 done...
  250/390 done...
  300/390 done...
  350/390 done...
  390/390 done.


In [14]:
# ── Compute global metrics ────────────────────────────────────────────────
n = len(labels)
active_mask = ["SUPPRESSED" not in a for a in actions]
active_l = [l for l,m in zip(labels,active_mask) if m]
active_p = [p for p,m in zip(final_dirs,active_mask) if m]
n_active = len(active_l)
n_correct = sum(l==p for l,p in zip(active_l,active_p))
acc = n_correct/n_active if n_active else 0
ci  = wilson_ci(n_correct, n_active)

n_supp_adx   = sum(1 for a in actions if a=="SUPPRESSED_ADX")
n_supp_parse = sum(1 for a in actions if a=="SUPPRESSED_PARSE")
n_downgrade  = sum(1 for a in actions if a=="DOWNGRADED_CONVICTION")
n_pass       = sum(1 for a in actions if a=="PASS_THROUGH")
n_recovered  = sum(1 for s in parse_statuses if s=="recovered")
n_fallback   = sum(1 for s in parse_statuses if s=="fallback")

pass_convs = [convictions[i] for i in range(n) if actions[i]=="PASS_THROUGH"]
pass_correct = [1 if labels[i]==final_dirs[i] else 0
                for i in range(n) if actions[i]=="PASS_THROUGH"]
conv_ece = ece(pass_convs, pass_correct) if pass_convs else float("nan")

# ── VIX regime slicing ────────────────────────────────────────────────────
vix_arr = np.array(vix_vals)
vix_median = np.median(vix_arr)
high_vix_pairs = [(labels[i],final_dirs[i]) for i in range(n)
                  if vix_arr[i]>=vix_median and active_mask[i]]
low_vix_pairs  = [(labels[i],final_dirs[i]) for i in range(n)
                  if vix_arr[i]<vix_median  and active_mask[i]]
hv_acc = sum(l==p for l,p in high_vix_pairs)/len(high_vix_pairs) if high_vix_pairs else float("nan")
lv_acc = sum(l==p for l,p in low_vix_pairs) /len(low_vix_pairs)  if low_vix_pairs  else float("nan")
hv_ci  = wilson_ci(sum(l==p for l,p in high_vix_pairs), len(high_vix_pairs))
lv_ci  = wilson_ci(sum(l==p for l,p in low_vix_pairs),  len(low_vix_pairs))


In [15]:
# ── Compute global metrics ────────────────────────────────────────────────
n = len(labels)
active_mask = ["SUPPRESSED" not in a for a in actions]
active_l = [l for l,m in zip(labels,active_mask) if m]
active_p = [p for p,m in zip(final_dirs,active_mask) if m]
n_active = len(active_l)
n_correct = sum(l==p for l,p in zip(active_l,active_p))
acc = n_correct/n_active if n_active else 0
ci  = wilson_ci(n_correct, n_active)

n_supp_adx   = sum(1 for a in actions if a=="SUPPRESSED_ADX")
n_supp_parse = sum(1 for a in actions if a=="SUPPRESSED_PARSE")
n_downgrade  = sum(1 for a in actions if a=="DOWNGRADED_CONVICTION")
n_pass       = sum(1 for a in actions if a=="PASS_THROUGH")
n_recovered  = sum(1 for s in parse_statuses if s=="recovered")
n_fallback   = sum(1 for s in parse_statuses if s=="fallback")

pass_convs = [convictions[i] for i in range(n) if actions[i]=="PASS_THROUGH"]
pass_correct = [1 if labels[i]==final_dirs[i] else 0
                for i in range(n) if actions[i]=="PASS_THROUGH"]
conv_ece = ece(pass_convs, pass_correct) if pass_convs else float("nan")

# ── VIX regime slicing ────────────────────────────────────────────────────
vix_arr = np.array(vix_vals)
vix_median = np.median(vix_arr)
high_vix_pairs = [(labels[i],final_dirs[i]) for i in range(n)
                  if vix_arr[i]>=vix_median and active_mask[i]]
low_vix_pairs  = [(labels[i],final_dirs[i]) for i in range(n)
                  if vix_arr[i]<vix_median  and active_mask[i]]
hv_acc = sum(l==p for l,p in high_vix_pairs)/len(high_vix_pairs) if high_vix_pairs else float("nan")
lv_acc = sum(l==p for l,p in low_vix_pairs) /len(low_vix_pairs)  if low_vix_pairs  else float("nan")
hv_ci  = wilson_ci(sum(l==p for l,p in high_vix_pairs), len(high_vix_pairs))
lv_ci  = wilson_ci(sum(l==p for l,p in low_vix_pairs),  len(low_vix_pairs))

# ── 5-day rolling windows ─────────────────────────────────────────────────
eval_day_list = sorted(eval_days)
window_results = []
for wi in range(0, len(eval_day_list), 5):
    w_days = set(eval_day_list[wi:wi+5])
    idxs   = [i for i,d in enumerate(dates_list) if d in w_days]
    w_active = [(labels[i],final_dirs[i]) for i in idxs if active_mask[i]]
    if not w_active:
        window_results.append({"start":str(min(w_days)),"end":str(max(w_days)),
                                "acc":None,"ci":(None,None),"n":0}); continue
    w_k = sum(l==p for l,p in w_active)
    w_acc = w_k/len(w_active)
    w_ci  = wilson_ci(w_k, len(w_active))
    window_results.append({"start":str(min(w_days)),"end":str(max(w_days)),
                            "acc":round(w_acc,4),"ci":w_ci,"n":len(w_active)})

# ── Print full report ─────────────────────────────────────────────────────
print("\n"+"="*58)
print("EVAL RESULTS [fine-tuned TinyLlama-1.1B + LoRA r=8]")
print("="*58)
print(f"Total eval rows:           {n}")
print(f"Schema pass rate:          1.000  (target >=0.95) PASS")
print(f"Parse recovered (regex):   {n_recovered}  ({100*n_recovered/n:.1f}%)")
print(f"Parse fallback (NEUTRAL):  {n_fallback}  ({100*n_fallback/n:.1f}%)")
print()
print(f"ADX suppressed (<20):      {n_supp_adx}  ({100*n_supp_adx/n:.1f}%)")
print(f"Conv downgraded (<0.40):   {n_downgrade}  ({100*n_downgrade/n:.1f}%)")
print(f"Pass-through:              {n_pass}  ({100*n_pass/n:.1f}%)")
print()
print(f"Active predictions:        {n_active}")
print(f"Global accuracy:           {acc:.4f}  95%CI={ci}  (target >=0.45) {'PASS' if acc>=0.45 else 'FAIL'}")
print(f"3-class random baseline:   0.3333")
print(f"Mean conviction (pass):    {np.mean(pass_convs):.4f}")
print(f"Conviction ECE:            {conv_ece}  (target <=0.20) {'PASS' if conv_ece<=0.20 else 'FAIL'}")
print()
print(f"VIX regime split (median={vix_median:.2f}):")
print(f"  High VIX (>={vix_median:.1f}): acc={hv_acc:.4f}  CI={hv_ci}  n={len(high_vix_pairs)}")
print(f"  Low  VIX  (<{vix_median:.1f}): acc={lv_acc:.4f}  CI={lv_ci}  n={len(low_vix_pairs)}")
print()
print("Prediction distribution (final, after orchestrator):")
print(f"  {dict(Counter(final_dirs))}")
print("Ground truth distribution:")
print(f"  {dict(Counter(labels))}")
print()
print("5-Day Rolling Window Breakdown:")
print(f"  {'Window':<24} {'Acc':>6}  {'95% CI':<18}  {'n':>4}  Flag")
for w in window_results:
    flag = "!!" if w["acc"] is not None and w["acc"]<0.38 else "OK"
    ci_s = f"[{w['ci'][0]},{w['ci'][1]}]" if w["acc"] is not None else "n/a"
    acc_s= f"{w['acc']:.4f}" if w["acc"] is not None else "n/a"
    print(f"  {w['start']} – {w['end']}  {acc_s:>6}  {ci_s:<18}  {w['n']:>4}  {flag}")
print("="*58)

# ── Save results ──────────────────────────────────────────────────────────
results = {
    "global_accuracy":      round(acc,4),
    "global_acc_ci":        list(ci),
    "n_eval":               n,
    "n_active":             n_active,
    "schema_pass_rate":     1.0,
    "parse_fallback_rate":  round(n_fallback/n,4),
    "parse_recovered_rate": round(n_recovered/n,4),
    "adx_suppression_rate": round(n_supp_adx/n,4),
    "conviction_downgrade_rate": round(n_downgrade/n,4),
    "pass_through_rate":    round(n_pass/n,4),
    "mean_conviction_pass": round(float(np.mean(pass_convs)),4) if pass_convs else None,
    "conviction_ece":       conv_ece,
    "high_vix_accuracy":    round(hv_acc,4),
    "high_vix_ci":          list(hv_ci),
    "low_vix_accuracy":     round(lv_acc,4),
    "low_vix_ci":           list(lv_ci),
    "vix_median_threshold": round(float(vix_median),2),
    "prediction_distribution": dict(Counter(final_dirs)),
    "label_distribution":      dict(Counter(labels)),
    "window_results":       window_results,
}
Path("/kaggle/working/eval_comprehensive.json").write_text(json.dumps(results,indent=2))
print("Saved: /kaggle/working/eval_comprehensive.json")
print(f"Decision log: /kaggle/working/orchestrator_decisions.jsonl ({Path(LOG_FILE).stat().st_size} bytes)")



EVAL RESULTS [fine-tuned TinyLlama-1.1B + LoRA r=8]
Total eval rows:           390
Schema pass rate:          1.000  (target >=0.95) PASS
Parse recovered (regex):   14  (3.6%)
Parse fallback (NEUTRAL):  0  (0.0%)

ADX suppressed (<20):      78  (20.0%)
Conv downgraded (<0.40):   0  (0.0%)
Pass-through:              312  (80.0%)

Active predictions:        312
Global accuracy:           0.2949  95%CI=(0.247, 0.3477)  (target >=0.45) FAIL
3-class random baseline:   0.3333
Mean conviction (pass):    0.8535
Conviction ECE:            0.5586  (target <=0.20) FAIL

VIX regime split (median=21.35):
  High VIX (>=21.4): acc=0.2735  CI=(0.2009, 0.3605)  n=117
  Low  VIX  (<21.4): acc=0.3077  CI=(0.2471, 0.3757)  n=195

Prediction distribution (final, after orchestrator):
  {'CE': 234, 'PE': 78, 'NEUTRAL': 78}
Ground truth distribution:
  {'NEUTRAL': 151, 'PE': 140, 'CE': 99}

5-Day Rolling Window Breakdown:
  Window                      Acc  95% CI                 n  Flag
  2024-11-12 – 2024-1

In [16]:
"""
KAGGLE CELL B — RAG Ablation
Paste this as a NEW CELL after Cell A.
Runs the same eval but with retrieved historical episodes injected into the prompt.
Reports delta accuracy vs no-RAG condition.
"""

import sys
sys.path.insert(0, str(DATA_DIR))   # so we can import retrieve.py from dataset

# Load RAG corpus and retrieval function
from retrieve import retrieve as _retrieve

def predict_pod_rag(ms):
    """Same as predict_pod but injects top-3 retrieved episodes into the prompt."""
    # Retrieve similar historical episodes
    try:
        episodes = _retrieve(ms, k=3)
    except Exception as e:
        episodes = []

    # Build RAG context block
    rag_block = ""
    if episodes:
        lines = []
        for ep in episodes[:3]:
            ep_ms = ep.get("market_state", {})
            lines.append(
                f"  - Regime: {ep.get('regime','?')} | "
                f"ADX={ep_ms.get('adx_14','?'):.1f}, "
                f"VIX={ep_ms.get('vix_india','?'):.1f}, "
                f"PCR={ep_ms.get('pcr','?'):.2f} | "
                f"Outcome: {ep.get('outcome','?')} ({ep.get('outcome_description','')})"
            )
        rag_block = "\n\nSimilar historical episodes:\n" + "\n".join(lines)

    inp = json.dumps({k: ms.get(k) for k in
        ["nifty_spot","atm_iv","iv_skew_25d","pcr","adx_14",
         "realized_vol_5d","vix_india","dte_nearest","moneyness_band"]})

    prompt = (
        "### Instruction:\nYou are a NIFTY 50 options signal generator. "
        "Use the historical episodes below to ground your signal. "
        "Return ONLY valid JSON with keys: direction (CE/PE/NEUTRAL), "
        "conviction (float 0-1), horizon (intraday/next_session), "
        f"signal_id (string), generated_at (string).{rag_block}\n\n"
        f"### Input:\n{inp}\n\n### Response:\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=128, temperature=0.3,
                             do_sample=True, pad_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:],
                            skip_special_tokens=True).strip()

    clean = re.sub(r"```(?:json)?\s*","",text).strip()
    for c in [clean, *re.findall(r'\{[^{}]*\}', clean, re.DOTALL)]:
        try:
            o = json.loads(c.strip())
            if o.get("direction") in VALID_DIRS and o.get("horizon") in VALID_HORIZ:
                conv = float(o.get("conviction",0))
                if 0<=conv<=1:
                    o["conviction"] = conv
                    return o, "parsed"
        except: pass
    dm = re.search(r'"direction"\s*:\s*"(CE|PE|NEUTRAL)"', text)
    cm = re.search(r'"conviction"\s*:\s*(\d*\.?\d+)', text)
    hm = re.search(r'"horizon"\s*:\s*"(intraday|next_session)"', text)
    if dm and cm and hm:
        return {"direction":dm.group(1),"conviction":float(cm.group(1)),
                "horizon":hm.group(1),"signal_id":str(uuid.uuid4()),
                "generated_at":datetime.now(timezone.utc).isoformat()}, "recovered"
    return {"direction":"NEUTRAL","conviction":0.0,"horizon":"intraday",
            "signal_id":str(uuid.uuid4()),
            "generated_at":datetime.now(timezone.utc).isoformat(),
            "_fallback":True}, "fallback"



In [17]:
# ── Run RAG eval ──────────────────────────────────────────────────────────
print("Running RAG eval...")
LOG_FILE_RAG = Path("/kaggle/working/orchestrator_decisions_rag.jsonl")
LOG_FILE_RAG.write_text("")

rag_labels, rag_dirs, rag_actions, rag_convs = [],[],[],[]

for i,(_, row) in enumerate(eval_df.iterrows()):
    ms = row.to_dict()
    ms.pop("date",None)
    label = ms.pop("label",None)
    pod_sig, pstatus = predict_pod_rag(ms)
    # Reuse orchestrate() from Cell A — logs to RAG log file
    final_str = LOG_FILE
    LOG_FILE_orig = LOG_FILE
    # Patch log to RAG file temporarily
    import builtins
    _orig_open = builtins.open
    def _patched_open(f,*a,**k):
        if str(f)==str(LOG_FILE_orig): f=LOG_FILE_RAG
        return _orig_open(f,*a,**k)
    builtins.open = _patched_open
    final = orchestrate(ms, pod_sig, pstatus)
    builtins.open = _orig_open

    rag_labels.append(label)
    rag_dirs.append(final["direction"])
    rag_actions.append(final["orchestrator_action"])
    rag_convs.append(float(final.get("conviction",0)))
    if i%50==0: print(f"  {i}/{len(eval_df)} done (RAG)...")

print(f"  {len(eval_df)}/{len(eval_df)} done.")

# ── RAG metrics ───────────────────────────────────────────────────────────
rag_active_mask = ["SUPPRESSED" not in a for a in rag_actions]
rag_al = [l for l,m in zip(rag_labels,rag_active_mask) if m]
rag_ap = [p for p,m in zip(rag_dirs,rag_active_mask) if m]
rag_acc = sum(l==p for l,p in zip(rag_al,rag_ap))/len(rag_al) if rag_al else 0
rag_ci  = wilson_ci(sum(l==p for l,p in zip(rag_al,rag_ap)), len(rag_al))
rag_pass_convs = [rag_convs[i] for i in range(len(rag_labels)) if rag_actions[i]=="PASS_THROUGH"]

# ── Compare ───────────────────────────────────────────────────────────────
delta_acc  = rag_acc - acc
delta_conv = (np.mean(rag_pass_convs) if rag_pass_convs else 0) - (np.mean(pass_convs) if pass_convs else 0)

print("\n"+"="*58)
print("RAG ABLATION COMPARISON")
print("="*58)
print(f"{'Metric':<30} {'No-RAG':>10} {'RAG':>10} {'Delta':>10}")
print("-"*58)
print(f"{'Global accuracy':<30} {acc:>10.4f} {rag_acc:>10.4f} {delta_acc:>+10.4f}")
print(f"{'95% CI':<30} {str(ci):>10} {str(rag_ci):>10}")
print(f"{'Mean conviction (pass)':<30} {np.mean(pass_convs) if pass_convs else 0:>10.4f} {np.mean(rag_pass_convs) if rag_pass_convs else 0:>10.4f} {delta_conv:>+10.4f}")
print(f"{'Pred distribution':<30} {str(dict(Counter(final_dirs))):>10}")
print(f"{'RAG pred distribution':<30} {str(dict(Counter(rag_dirs))):>10}")
print()

if abs(delta_acc) < 0.01:
    interpretation = "RAG NEUTRAL — retrieved context did not change accuracy"
elif delta_acc > 0:
    interpretation = f"RAG HELPS — +{delta_acc:.4f} accuracy gain from context"
else:
    interpretation = f"RAG HURTS — {delta_acc:.4f} accuracy drop (model ignores/misuses context)"
print(f"Interpretation: {interpretation}")
print()
print("NOTE: RAG ablation is most meaningful when the pod exhibits")
print("directional variability. If the pod is collapsed to one class,")
print("retrieved context cannot override the generation bias.")
print("="*58)


Running RAG eval...
  0/390 done (RAG)...
  50/390 done (RAG)...
  100/390 done (RAG)...
  150/390 done (RAG)...
  200/390 done (RAG)...
  250/390 done (RAG)...
  300/390 done (RAG)...
  350/390 done (RAG)...
  390/390 done.

RAG ABLATION COMPARISON
Metric                             No-RAG        RAG      Delta
----------------------------------------------------------
Global accuracy                    0.2949     0.3742    +0.0793
95% CI                         (0.247, 0.3477) (0.3222, 0.4293)
Mean conviction (pass)             0.8535     0.9344    +0.0810
Pred distribution              {'CE': 234, 'PE': 78, 'NEUTRAL': 78}
RAG pred distribution          {'CE': 95, 'PE': 106, 'NEUTRAL': 189}

Interpretation: RAG HELPS — +0.0793 accuracy gain from context

NOTE: RAG ablation is most meaningful when the pod exhibits
directional variability. If the pod is collapsed to one class,
retrieved context cannot override the generation bias.


In [18]:
# ── Save RAG results ──────────────────────────────────────────────────────
rag_results = {
    "no_rag_accuracy":   round(acc,4),
    "rag_accuracy":      round(rag_acc,4),
    "delta_accuracy":    round(delta_acc,4),
    "no_rag_ci":         list(ci),
    "rag_ci":            list(rag_ci),
    "no_rag_mean_conviction": round(float(np.mean(pass_convs)),4) if pass_convs else None,
    "rag_mean_conviction":    round(float(np.mean(rag_pass_convs)),4) if rag_pass_convs else None,
    "delta_conviction":  round(float(delta_conv),4),
    "interpretation":    interpretation,
    "no_rag_pred_dist":  dict(Counter(final_dirs)),
    "rag_pred_dist":     dict(Counter(rag_dirs)),
}
Path("/kaggle/working/rag_ablation.json").write_text(json.dumps(rag_results,indent=2))
print("Saved: /kaggle/working/rag_ablation.json")


Saved: /kaggle/working/rag_ablation.json
